# Chapter 10: Convolutional and Recurrent Networks
**Part III — Deep Learning**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

CNNs, ResNet, RNNs, LSTMs, GRUs, and attention mechanisms.

## Learning Objectives
See Chapter 10 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## A simple CNN for image classification


In [ ]:
import torch.nn as nn

# A simple CNN for image classification
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),              # 16x16
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4))   # fixed spatial output
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256),
            nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

##         # Project and split into heads


In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B, T, D = q.shape
        # Project and split into heads
        Q = self.W_q(q).view(B, T, self.n_heads, self.d_k).transpose(1,2)
        K = self.W_k(k).view(B, T, self.n_heads, self.d_k).transpose(1,2)
        V = self.W_v(v).view(B, T, self.n_heads, self.d_k).transpose(1,2)
        # Scaled dot-product attention
        scores = (Q @ K.transpose(-2,-1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask==0, -1e9)
        weights = scores.softmax(dim=-1)
        out = (weights @ V).transpose(1,2).reshape(B, T, D)
        return self.W_o(out)

## Visualising CNN filters and feature maps


In [ ]:
# Visualising CNN filters and feature maps
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

# Load pretrained ResNet-50 and inspect first conv layer
model = models.resnet50(weights='IMAGENET1K_V2')
model.eval()

# First convolutional layer: 64 filters of size 7×7×3
first_conv = model.conv1.weight.data.cpu().numpy()
print(f"First conv layer shape: {first_conv.shape}")
print(f"  64 filters, each 7×7, applied to 3 colour channels")

# Visualise the 64 filters (normalised for display)
fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for i, ax in enumerate(axes.ravel()):
    if i >= 64: break
    filt = first_conv[i]
    filt_norm = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
    # Average across channels for greyscale display
    ax.imshow(filt_norm.mean(axis=0), cmap='viridis', vmin=0, vmax=1)
    ax.axis('off')

plt.suptitle('ResNet-50: All 64 filters in the first convolutional layer\n'
             '(7×7 spatial kernels — each detects a different edge/texture pattern)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ch10_cnn_filters.png', dpi=150, bbox_inches='tight')
plt.show()
print("These filters are learned, not hand-crafted — they emerge from training on ImageNet.")

---

## 📚 Review Questions

See Chapter 10 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 10 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*